# Phase 5: Chấm điểm đích thuốc ứng viên

Notebook này gộp bằng chứng biểu hiện từ DEG và đặc trưng mạng protein từ Phase 4, chuẩn hóa các đặc trưng về thang 0-1, tính điểm cuối cùng và xuất top 100 đích thuốc ứng viên.

Nguyên tắc chạy:
- Chỉ đọc output Phase 3 và Phase 4 từ HDFS.
- Không sửa raw/refined.
- Ghi bảng đầy đủ vào `analysis/candidate_target_features`.
- Ghi top 100 vào `mart/top_candidate_targets`.

In [1]:
from pyspark.sql import SparkSession, Window, functions as F

# Khai báo đường dẫn HDFS cho Phase 5.
HDFS_BASE = "hdfs://master11:9000"
DEG_PROTEINS_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/deg_mapped_proteins"
NETWORK_FEATURES_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/protein_network_features"
CANDIDATE_FEATURES_OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/analysis/candidate_target_features"
TOP_TARGETS_OUTPUT_PATH = f"{HDFS_BASE}/drugtarget/data/mart/top_candidate_targets"
TOP_N = 100

# Tạo SparkSession để tính điểm candidate target.
spark = (
    SparkSession.builder.appName("drugtarget-gdc-phase5-candidate-target-scoring")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .enableHiveSupport()
    .getOrCreate()
)

# Giảm log để output notebook dễ đọc.
spark.sparkContext.setLogLevel("WARN")

print(f"Spark master: {spark.sparkContext.master}")
print(f"Input DEG proteins: {DEG_PROTEINS_PATH}")
print(f"Input network features: {NETWORK_FEATURES_PATH}")
print(f"Output candidate features: {CANDIDATE_FEATURES_OUTPUT_PATH}")
print(f"Output top {TOP_N}: {TOP_TARGETS_OUTPUT_PATH}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/03 08:18:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark master: local[*]
Input DEG proteins: hdfs://master11:9000/drugtarget/data/analysis/deg_mapped_proteins
Input network features: hdfs://master11:9000/drugtarget/data/analysis/protein_network_features
Output candidate features: hdfs://master11:9000/drugtarget/data/analysis/candidate_target_features
Output top 100: hdfs://master11:9000/drugtarget/data/mart/top_candidate_targets


In [2]:
def require_columns(frame, required_columns, frame_name: str) -> None:
    """Dừng pipeline nếu DataFrame thiếu cột bắt buộc."""
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        missing_text = ", ".join(missing_columns)
        raise ValueError(f"{frame_name} thiếu cột bắt buộc: {missing_text}")


def minmax_col(col_name: str, min_val, max_val):
    """Chuẩn hóa min-max an toàn về thang 0-1."""
    if min_val is None or max_val is None or float(max_val) == float(min_val):
        return F.lit(0.0)
    return (F.col(col_name) - F.lit(float(min_val))) / (F.lit(float(max_val)) - F.lit(float(min_val)))


# Đọc input Phase 5.
deg_proteins = spark.read.parquet(DEG_PROTEINS_PATH)
protein_features = spark.read.parquet(NETWORK_FEATURES_PATH)

DEG_REQUIRED_COLUMNS = [
    "gene_id_base",
    "gene_name",
    "protein_id",
    "ensp_id",
    "log2FC",
    "p_value",
    "deg_direction",
]
NETWORK_REQUIRED_COLUMNS = [
    "protein_id",
    "protein_name",
    "degree_protein",
    "weighted_degree_protein",
    "num_interactions_in_deg_network",
    "avg_combined_score",
]
require_columns(deg_proteins, DEG_REQUIRED_COLUMNS, "deg_mapped_proteins")
require_columns(protein_features, NETWORK_REQUIRED_COLUMNS, "protein_network_features")

print("Schema deg_mapped_proteins:")
deg_proteins.printSchema()
print("Schema protein_network_features:")
protein_features.printSchema()

Schema deg_mapped_proteins:
root
 |-- gene_id_base: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- gene_confidence: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)

Schema protein_network_features:
root
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- gene_name: string (nullable = true)
 |-- degree_protein: long (nullable = true)
 |-- weighted_degree_protein: double (nullable = true)
 |-- num_interactions_in_deg_network: long (nullable = true)
 |-- avg_combined_score: double (nullable = true)
 |-- max_combined_score: double (nullable = true)
 |-- num_high_confidence_edges: long (nullable = true)



In [3]:
# Join bằng protein_id để tạo feature tổng hợp cho từng gene/protein candidate.
candidate_target_features_raw = (
    deg_proteins.alias("d")
    .join(protein_features.alias("n"), on="protein_id", how="left")
    .select(
        F.col("d.gene_id_base"),
        F.col("d.gene_name"),
        F.col("d.protein_id"),
        F.col("d.ensp_id"),
        F.col("n.protein_name"),
        F.col("d.log2FC").cast("double").alias("log2FC"),
        F.col("d.p_value").cast("double").alias("p_value"),
        F.col("d.deg_direction"),
        F.coalesce(F.col("n.degree_protein"), F.lit(0)).cast("long").alias("degree_protein"),
        F.coalesce(F.col("n.weighted_degree_protein"), F.lit(0.0)).cast("double").alias("weighted_degree_protein"),
        F.coalesce(F.col("n.num_interactions_in_deg_network"), F.lit(0)).cast("long").alias("num_interactions_in_deg_network"),
        F.coalesce(F.col("n.avg_combined_score"), F.lit(0.0)).cast("double").alias("avg_combined_score"),
    )
    .filter(F.col("gene_name").isNotNull() & F.col("protein_id").isNotNull())
    .withColumn("abs_log2FC", F.abs(F.col("log2FC")))
)

# Nếu một protein có nhiều evidence, giữ evidence có abs(log2FC) lớn nhất rồi p_value thấp nhất để rank không bị trùng protein.
evidence_window = Window.partitionBy("protein_id").orderBy(
    F.desc("abs_log2FC"),
    F.asc_nulls_last("p_value"),
    F.asc("gene_name"),
)
candidate_target_features = (
    candidate_target_features_raw.withColumn("evidence_rank", F.row_number().over(evidence_window))
    .filter(F.col("evidence_rank") == 1)
    .drop("evidence_rank")
    .cache()
)

num_candidates = candidate_target_features.count()
if num_candidates == 0:
    raise ValueError("Không có candidate target để chấm điểm.")
print(f"Số candidate protein để chấm điểm: {num_candidates}")

# Lấy min/max để chuẩn hóa các feature về thang 0-1.
stats = candidate_target_features.agg(
    F.min("abs_log2FC").alias("min_abs_log2FC"),
    F.max("abs_log2FC").alias("max_abs_log2FC"),
    F.min("weighted_degree_protein").alias("min_weighted_degree"),
    F.max("weighted_degree_protein").alias("max_weighted_degree"),
    F.min("avg_combined_score").alias("min_avg_combined_score"),
    F.max("avg_combined_score").alias("max_avg_combined_score"),
).collect()[0]

# Tính score thành phần và final_score theo trọng số đã chốt.
candidate_scored = (
    candidate_target_features.withColumn(
        "expression_score",
        minmax_col("abs_log2FC", stats["min_abs_log2FC"], stats["max_abs_log2FC"]),
    )
    .withColumn(
        "protein_network_score",
        minmax_col("weighted_degree_protein", stats["min_weighted_degree"], stats["max_weighted_degree"]),
    )
    .withColumn(
        "string_confidence_score",
        minmax_col("avg_combined_score", stats["min_avg_combined_score"], stats["max_avg_combined_score"]),
    )
    .withColumn(
        "final_score",
        F.lit(0.5) * F.col("expression_score")
        + F.lit(0.3) * F.col("protein_network_score")
        + F.lit(0.2) * F.col("string_confidence_score"),
    )
)

# Rank ổn định để top target tái lập được giữa các lần chạy.
rank_window = Window.orderBy(
    F.desc("final_score"),
    F.desc("abs_log2FC"),
    F.desc("weighted_degree_protein"),
    F.asc("gene_name"),
)

OUTPUT_COLUMNS = [
    "rank",
    "gene_name",
    "gene_id_base",
    "protein_id",
    "ensp_id",
    "protein_name",
    "log2FC",
    "p_value",
    "deg_direction",
    "degree_protein",
    "weighted_degree_protein",
    "num_interactions_in_deg_network",
    "avg_combined_score",
    "expression_score",
    "protein_network_score",
    "string_confidence_score",
    "final_score",
]

candidate_ranked = (
    candidate_scored.withColumn("rank", F.row_number().over(rank_window))
    .select(*OUTPUT_COLUMNS)
    .cache()
)

top_candidate_targets = candidate_ranked.filter(F.col("rank") <= F.lit(TOP_N)).cache()

print("Schema output Phase 5:")
candidate_ranked.printSchema()
print(f"Số dòng candidate_target_features: {candidate_ranked.count()}")
print(f"Số dòng top_candidate_targets: {top_candidate_targets.count()}")

Số candidate protein để chấm điểm: 2579


26/06/03 08:19:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Schema output Phase 5:
root
 |-- rank: integer (nullable = false)
 |-- gene_name: string (nullable = true)
 |-- gene_id_base: string (nullable = true)
 |-- protein_id: string (nullable = true)
 |-- ensp_id: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- log2FC: double (nullable = true)
 |-- p_value: double (nullable = true)
 |-- deg_direction: string (nullable = true)
 |-- degree_protein: long (nullable = false)
 |-- weighted_degree_protein: double (nullable = false)
 |-- num_interactions_in_deg_network: long (nullable = false)
 |-- avg_combined_score: double (nullable = false)
 |-- expression_score: double (nullable = true)
 |-- protein_network_score: double (nullable = true)
 |-- string_confidence_score: double (nullable = true)
 |-- final_score: double (nullable = true)



26/06/03 08:19:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/06/03 08:19:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Số dòng candidate_target_features: 2579


Số dòng top_candidate_targets: 100


In [4]:
# Ghi bảng candidate đầy đủ và top 100 ra HDFS.
candidate_ranked.write.mode("overwrite").option("compression", "snappy").parquet(CANDIDATE_FEATURES_OUTPUT_PATH)
top_candidate_targets.write.mode("overwrite").option("compression", "snappy").parquet(TOP_TARGETS_OUTPUT_PATH)
print(f"Đã ghi output Phase 5: {CANDIDATE_FEATURES_OUTPUT_PATH}")
print(f"Đã ghi top {TOP_N}: {TOP_TARGETS_OUTPUT_PATH}")

Đã ghi output Phase 5: hdfs://master11:9000/drugtarget/data/analysis/candidate_target_features
Đã ghi top 100: hdfs://master11:9000/drugtarget/data/mart/top_candidate_targets


In [5]:
# Kiểm tra output path tồn tại trên HDFS.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for output_path in [CANDIDATE_FEATURES_OUTPUT_PATH, TOP_TARGETS_OUTPUT_PATH]:
    hdfs_path = spark._jvm.org.apache.hadoop.fs.Path(output_path)
    fs = hdfs_path.getFileSystem(hadoop_conf)
    if not fs.exists(hdfs_path):
        raise AssertionError(f"Output Phase 5 chưa tồn tại trên HDFS: {output_path}")

# Đọc lại output để xác nhận schema và ràng buộc top 100.
candidate_output = spark.read.parquet(CANDIDATE_FEATURES_OUTPUT_PATH)
top_output = spark.read.parquet(TOP_TARGETS_OUTPUT_PATH)
if candidate_output.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema candidate_target_features không đúng: {candidate_output.columns}")
if top_output.columns != OUTPUT_COLUMNS:
    raise AssertionError(f"Schema top_candidate_targets không đúng: {top_output.columns}")

score_columns = ["expression_score", "protein_network_score", "string_confidence_score", "final_score"]
bad_score_count = candidate_output.filter(
    " OR ".join([f"{col} IS NULL OR {col} < 0 OR {col} > 1" for col in score_columns])
).count()
if bad_score_count != 0:
    raise AssertionError(f"Có {bad_score_count} dòng score ngoài khoảng [0, 1].")

top_count = top_output.count()
if top_count > TOP_N:
    raise AssertionError(f"Top output có quá {TOP_N} dòng: {top_count}")

null_top_id_count = top_output.filter(F.col("gene_name").isNull() | F.col("protein_id").isNull()).count()
if null_top_id_count != 0:
    raise AssertionError(f"Top output có {null_top_id_count} dòng thiếu gene_name/protein_id.")

rank_stats = candidate_output.agg(
    F.min("rank").alias("min_rank"),
    F.max("rank").alias("max_rank"),
    F.countDistinct("rank").alias("distinct_rank"),
    F.count("*").alias("num_rows"),
).collect()[0]
if rank_stats["min_rank"] != 1 or rank_stats["max_rank"] != rank_stats["num_rows"] or rank_stats["distinct_rank"] != rank_stats["num_rows"]:
    raise AssertionError(f"Rank không liên tục: {dict(rank_stats.asDict())}")

bad_order_count = (
    top_output.select("rank", "final_score")
    .withColumn("prev_final_score", F.lag("final_score").over(Window.orderBy("rank")))
    .filter(F.col("prev_final_score").isNotNull() & (F.col("final_score") > F.col("prev_final_score")))
    .count()
)
if bad_order_count != 0:
    raise AssertionError(f"Top output không sắp theo final_score giảm dần ở {bad_order_count} vị trí.")

print("Kiểm tra output Phase 5 hoàn tất.")
print(f"Số dòng top_candidate_targets: {top_count}")
print("Top 20 đích thuốc ứng viên:")
top_output.orderBy("rank").show(20, truncate=False)

26/06/03 08:19:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/03 08:19:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Kiểm tra output Phase 5 hoàn tất.
Số dòng top_candidate_targets: 100
Top 20 đích thuốc ứng viên:


+----+---------+---------------+--------------------+---------------+------------+-------------------+-----------------------+-------------+--------------+-----------------------+-------------------------------+------------------+-------------------+---------------------+-----------------------+-------------------+
|rank|gene_name|gene_id_base   |protein_id          |ensp_id        |protein_name|log2FC             |p_value                |deg_direction|degree_protein|weighted_degree_protein|num_interactions_in_deg_network|avg_combined_score|expression_score   |protein_network_score|string_confidence_score|final_score        |
+----+---------+---------------+--------------------+---------------+------------+-------------------+-----------------------+-------------+--------------+-----------------------+-------------------------------+------------------+-------------------+---------------------+-----------------------+-------------------+
|1   |SFTPC    |ENSG00000168484|9606.ENSP00000316

In [6]:
# Giải phóng cache sau khi hoàn tất Phase 5.
candidate_target_features.unpersist()
candidate_ranked.unpersist()
top_candidate_targets.unpersist()
print("Hoàn tất Phase 5: Chấm điểm đích thuốc ứng viên")

Hoàn tất Phase 5: Chấm điểm đích thuốc ứng viên
